# hg38 lncRNA filtering: GENCODE basic analysis

Investigate the effect of filtering lncRNA transcripts to only GENCODE basic.

- How many lncRNA transcripts do we have in total?
- How many of those are GENCODE basic?
- How many genes have at least one GENCODE basic transcript?
- What happens to union (collapsed) transcripts if we restrict to GENCODE basic?
- Total exon length comparison before vs after.

## Part I: BioMart-based analysis (sections 1-8)
## Part II: GTF tag-level analysis (sections 9-12)

In [1]:
import pandas as pd
import numpy as np
from pyrion.io.bed import read_bed12_file
from pyrion.io.gene_data import read_gene_data

## 1. Load transcripts and metadata

In [2]:
BED_PATH = "reference_annotation/hg38.input.w.tRNA.bed"
META_PATH = "reference_annotation/hg38.transcript_metadata.tsv"
MART_PATH = "reference_annotation/mart_export.txt"

transcripts = read_bed12_file(BED_PATH)
gene_data = read_gene_data(
    META_PATH,
    gene_column="gene_id",
    transcript_id_column="transcript_id",
    transcript_type_column="transcript_biotype",
)
transcripts.bind_gene_data(gene_data)
print(transcripts.summary())

386,300 transcripts, 79,317 genes from reference_annotation/hg38.input.w.tRNA.bed


## 2. Parse BioMart export — GENCODE basic lncRNA set

In [3]:
mart = pd.read_csv(MART_PATH, sep="\t")
mart.head()

,Gene stable ID,GENCODE basic annotation,Transcript support level (TSL),Gene name,GENCODE primary annotation,APPRIS annotation,Ensembl Canonical,Transcript stable ID
0,ENSG00000299200,NaN,NaN,NaN,GENCODE primary,NaN,1.0,ENST00000761542
1,ENSG00000299200,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000761543
2,ENSG00000299200,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000761544
3,ENSG00000299200,GENCODE basic,NaN,NaN,NaN,NaN,NaN,ENST00000761545
4,ENSG00000299200,NaN,NaN,NaN,NaN,NaN,NaN,ENST00000761546


In [4]:
gencode_basic_transcripts = set(
    mart.loc[
        mart["GENCODE basic annotation"] == "GENCODE basic",
        "Transcript stable ID",
    ]
)
print(f"Total transcripts in BioMart export: {len(mart):,}")
print(f"GENCODE basic transcripts (unversioned): {len(gencode_basic_transcripts):,}")

Total transcripts in BioMart export: 195,159
GENCODE basic transcripts (unversioned): 72,074


## 3. Separate lncRNAs from everything else

In [5]:
lncrna_coll = transcripts.filter_by_biotype("lncRNA")
non_lncrna_ids = set(t.id for t in transcripts) - set(t.id for t in lncrna_coll)

n_total = len(transcripts)
n_lncrna = len(lncrna_coll)
n_other = len(non_lncrna_ids)

print(f"Total transcripts:     {n_total:>8,}")
print(f"lncRNA transcripts:    {n_lncrna:>8,}")
print(f"Non-lncRNA transcripts:{n_other:>8,}")

Total transcripts:      386,300
lncRNA transcripts:     189,149
Non-lncRNA transcripts: 197,151


## 4. lncRNA genes overview

In [6]:
lncrna_genes = lncrna_coll.genes
n_lncrna_genes = len(lncrna_genes)
print(f"lncRNA genes: {n_lncrna_genes:,}")

isoforms_per_gene = [len(g.transcripts) for g in lncrna_genes]
print(f"Isoforms per gene — median: {np.median(isoforms_per_gene):.0f}, "
      f"mean: {np.mean(isoforms_per_gene):.1f}, "
      f"max: {max(isoforms_per_gene)}")

lncRNA genes: 34,846
Isoforms per gene — median: 1, mean: 5.4, max: 522


## 5. Which lncRNA transcripts are GENCODE basic?

In [7]:
def strip_version(tid: str) -> str:
    return tid.rsplit(".", 1)[0]

lncrna_basic_ids = set()
lncrna_non_basic_ids = set()

for t in lncrna_coll:
    if strip_version(t.id) in gencode_basic_transcripts:
        lncrna_basic_ids.add(t.id)
    else:
        lncrna_non_basic_ids.add(t.id)

print(f"lncRNA GENCODE basic:     {len(lncrna_basic_ids):>8,}")
print(f"lncRNA non-GENCODE basic: {len(lncrna_non_basic_ids):>8,}")
print(f"Fraction basic: {len(lncrna_basic_ids) / n_lncrna:.1%}")

lncRNA GENCODE basic:       69,478
lncRNA non-GENCODE basic:  119,671
Fraction basic: 36.7%


## 6. How many lncRNA genes have at least one GENCODE basic transcript?

In [8]:
genes_with_basic = 0
genes_without_basic = 0

for gene in lncrna_genes:
    has_basic = any(t.id in lncrna_basic_ids for t in gene.transcripts)
    if has_basic:
        genes_with_basic += 1
    else:
        genes_without_basic += 1

print(f"lncRNA genes with >= 1 GENCODE basic transcript:  {genes_with_basic:>6,}")
print(f"lncRNA genes with NO GENCODE basic transcript:    {genes_without_basic:>6,}")
print(f"Fraction of genes retained: {genes_with_basic / n_lncrna_genes:.1%}")

lncRNA genes with >= 1 GENCODE basic transcript:  34,832
lncRNA genes with NO GENCODE basic transcript:        14
Fraction of genes retained: 100.0%


## 7. Union (collapsed) transcripts: all lncRNAs vs GENCODE basic only

In [9]:
def compute_union_stats(genes):
    """For each gene, compute union transcript exon length."""
    total_exon_len = 0
    n_genes = 0
    for gene in genes:
        union_t = gene.to_union_transcript()
        exon_len = sum(int(b[1]) - int(b[0]) for b in union_t.blocks)
        total_exon_len += exon_len
        n_genes += 1
    return n_genes, total_exon_len


n_union_all, exon_len_all = compute_union_stats(lncrna_genes)
print(f"--- All lncRNA isoforms ---")
print(f"Union transcripts (genes):  {n_union_all:>8,}")
print(f"Total union exon length:    {exon_len_all:>12,} bp")

--- All lncRNA isoforms ---
Union transcripts (genes):    34,846
Total union exon length:      68,528,306 bp


In [10]:
from pyrion.core.genes import Gene, Transcript, TranscriptsCollection

basic_only_genes = []
for gene in lncrna_genes:
    basic_transcripts = [t for t in gene.transcripts if t.id in lncrna_basic_ids]
    if basic_transcripts:
        basic_only_genes.append(
            Gene(gene_id=gene.gene_id, transcripts=basic_transcripts, gene_name=gene.gene_name)
        )

n_union_basic, exon_len_basic = compute_union_stats(basic_only_genes)
print(f"--- GENCODE basic lncRNA isoforms only ---")
print(f"Union transcripts (genes):  {n_union_basic:>8,}")
print(f"Total union exon length:    {exon_len_basic:>12,} bp")

--- GENCODE basic lncRNA isoforms only ---
Union transcripts (genes):    34,832
Total union exon length:      58,269,540 bp


## 8. Summary comparison

In [11]:
raw_exon_len_all = sum(
    sum(int(b[1]) - int(b[0]) for b in t.blocks)
    for t in lncrna_coll
)
raw_exon_len_basic = sum(
    sum(int(b[1]) - int(b[0]) for b in t.blocks)
    for t in lncrna_coll if t.id in lncrna_basic_ids
)

summary = pd.DataFrame({
    "All lncRNA": [
        n_lncrna,
        n_lncrna_genes,
        n_union_all,
        f"{raw_exon_len_all:,}",
        f"{exon_len_all:,}",
    ],
    "GENCODE basic only": [
        len(lncrna_basic_ids),
        genes_with_basic,
        n_union_basic,
        f"{raw_exon_len_basic:,}",
        f"{exon_len_basic:,}",
    ],
}, index=[
    "Transcripts",
    "Genes",
    "Union transcripts (collapsed genes)",
    "Total raw exon length (bp)",
    "Total union exon length (bp)",
])
summary

,All lncRNA,GENCODE basic only
Transcripts,189149,69478
Genes,34846,34832
Union transcripts (collapsed genes),34846,34832
Total raw exon length (bp),"192,459,784","80,978,801"
Total union exon length (bp),"68,528,306","58,269,540"


---

## Part II: GTF tag-level analysis

Parse the GENCODE v48 GTF directly to extract per-transcript tags, level, and TSL
for **all transcripts from lncRNA genes** (`gene_type "lncRNA"`), not just those
with `transcript_type "lncRNA"`. This keeps retained_intron, processed_transcript,
etc. so we can evaluate a "primary only" strategy without biotype filtering.

In [12]:
import re
from collections import Counter

GTF_PATH = "/Users/Bogdan.Kirilenko/PycharmProjects/CURIA/input_data/reference_annotations/gencode.v48.annotation.gtf"

RUBBISH_TAGS = {"mRNA_start_NF", "mRNA_end_NF", "cds_start_NF", "cds_end_NF", "low_sequence_quality"}
TAG_RE = re.compile(r'tag "([^"]+)"')
LEVEL_RE = re.compile(r'level (\d+)')
TSL_RE = re.compile(r'transcript_support_level "([^"]+)"')
TID_RE = re.compile(r'transcript_id "([^"]+)"')
GID_RE = re.compile(r'gene_id "([^"]+)"')
GNAME_RE = re.compile(r'gene_name "([^"]+)"')
TTYPE_RE = re.compile(r'transcript_type "([^"]+)"')

records = []

with open(GTF_PATH) as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.split("\t")
        if fields[2] != "transcript":
            continue
        attrs = fields[8]
        # Filter by gene_type, NOT transcript_type — keeps retained_intron etc.
        if 'gene_type "lncRNA"' not in attrs:
            continue

        tid = TID_RE.search(attrs).group(1)
        gid = GID_RE.search(attrs).group(1)
        tags = set(TAG_RE.findall(attrs))

        gname_m = GNAME_RE.search(attrs)
        gene_name = gname_m.group(1) if gname_m else None

        ttype_m = TTYPE_RE.search(attrs)
        transcript_type = ttype_m.group(1) if ttype_m else None

        level_m = LEVEL_RE.search(attrs)
        level = int(level_m.group(1)) if level_m else None

        tsl_m = TSL_RE.search(attrs)
        tsl = tsl_m.group(1) if tsl_m else None

        records.append({
            "transcript_id": tid,
            "gene_id": gid,
            "gene_name": gene_name,
            "transcript_type": transcript_type,
            "level": level,
            "tsl": tsl,
            "has_basic": "basic" in tags,
            "has_primary": "GENCODE_Primary" in tags,
            "has_canonical": "Ensembl_canonical" in tags,
            "has_rubbish": bool(tags & RUBBISH_TAGS),
            "rubbish_tags": tags & RUBBISH_TAGS,
            "all_tags": tags,
        })

gtf_df = pd.DataFrame(records)
print(f"Transcripts from lncRNA genes (by gene_type): {len(gtf_df):,}")
print(f"lncRNA genes: {gtf_df['gene_id'].nunique():,}")
print(f"\nTranscript biotype distribution within lncRNA genes:")
print(gtf_df["transcript_type"].value_counts().to_string())
gtf_df.head()

lncRNA transcripts parsed from GTF: 189,149
lncRNA genes: 34,846


,transcript_id,gene_id,level,tsl,has_basic,has_primary,has_canonical,has_rubbish,rubbish_tags,all_tags
0,ENST00000832824.1,ENSG00000290825.2,2,NaN,False,False,False,False,{},{TAGENE}
1,ENST00000832825.1,ENSG00000290825.2,2,NaN,False,False,False,False,{},{TAGENE}
2,ENST00000832826.1,ENSG00000290825.2,2,NaN,False,False,False,False,{},{TAGENE}
3,ENST00000832827.1,ENSG00000290825.2,2,NaN,False,False,False,False,{},{TAGENE}
4,ENST00000832828.1,ENSG00000290825.2,2,NaN,True,False,False,False,{},"{TAGENE, basic}"


## 9. Transcript-level tag overview

In [13]:
n = len(gtf_df)
print("=== Transcript-level flags ===")
print(f"{'has_basic':30s} {gtf_df['has_basic'].sum():>8,}  ({gtf_df['has_basic'].mean():.1%})")
print(f"{'has_primary':30s} {gtf_df['has_primary'].sum():>8,}  ({gtf_df['has_primary'].mean():.1%})")
print(f"{'has_canonical':30s} {gtf_df['has_canonical'].sum():>8,}  ({gtf_df['has_canonical'].mean():.1%})")
print(f"{'has_rubbish_tag':30s} {gtf_df['has_rubbish'].sum():>8,}  ({gtf_df['has_rubbish'].mean():.1%})")
print()
print("=== Rubbish tag breakdown ===")
rubbish_counts = Counter()
for tags in gtf_df["rubbish_tags"]:
    rubbish_counts.update(tags)
for tag, cnt in rubbish_counts.most_common():
    print(f"  {tag:30s} {cnt:>6,}")
print()
print("=== Level distribution ===")
print(gtf_df["level"].value_counts().sort_index().to_string())
print()
print("=== TSL distribution ===")
print(gtf_df["tsl"].value_counts().sort_index().to_string())

=== Transcript-level flags ===
has_basic                        69,521  (36.8%)
has_primary                      34,846  (18.4%)
has_canonical                    34,846  (18.4%)
has_rubbish_tag                   1,799  (1.0%)

=== Rubbish tag breakdown ===
  mRNA_start_NF                     909
  mRNA_end_NF                       861
  low_sequence_quality               39

=== Level distribution ===
level
1      2582
2    186460
3       107

=== TSL distribution ===
tsl
1     3117
2     5093
3     7448
4     2685
5     7008
NA    2794


## 10. Cross-tabulation: basic vs rubbish vs level

In [14]:
print("=== basic x rubbish ===")
xt = pd.crosstab(gtf_df["has_basic"], gtf_df["has_rubbish"],
                 rownames=["basic"], colnames=["rubbish"], margins=True)
print(xt)
print()

print("=== basic x level ===")
xt2 = pd.crosstab(gtf_df["has_basic"], gtf_df["level"],
                  rownames=["basic"], colnames=["level"], margins=True)
print(xt2)
print()

print("=== basic x primary ===")
xt3 = pd.crosstab(gtf_df["has_basic"], gtf_df["has_primary"],
                  rownames=["basic"], colnames=["primary"], margins=True)
print(xt3)

=== basic x rubbish ===
rubbish   False  True     All
basic                        
False    117888  1740  119628
True      69462    59   69521
All      187350  1799  189149

=== basic x level ===
level     1       2    3     All
basic                           
False  1140  118399   89  119628
True   1442   68061   18   69521
All    2582  186460  107  189149

=== basic x primary ===
primary   False   True     All
basic                         
False    113643   5985  119628
True      40660  28861   69521
All      154303  34846  189149


## 11. Gene-level aggregation: what stays under different filters?

In [15]:
gene_agg = gtf_df.groupby("gene_id").agg(
    n_transcripts=("transcript_id", "count"),
    any_basic=("has_basic", "any"),
    all_basic=("has_basic", "all"),
    any_primary=("has_primary", "any"),
    any_canonical=("has_canonical", "any"),
    any_rubbish=("has_rubbish", "any"),
    all_rubbish=("has_rubbish", "all"),
    n_basic=("has_basic", "sum"),
    n_rubbish=("has_rubbish", "sum"),
).reset_index()

n_genes = len(gene_agg)
print(f"Total lncRNA genes: {n_genes:,}")
print()

filters = {
    "No filter (all)":           gene_agg,
    "Gene has >= 1 basic":       gene_agg[gene_agg["any_basic"]],
    "Gene has >= 1 non-rubbish": gene_agg[~gene_agg["all_rubbish"]],
    "Gene has >= 1 basic AND no rubbish on basic": None,  # computed below
}

# For the combined filter: genes where at least one transcript is basic AND not rubbish
basic_no_rubbish = gtf_df[gtf_df["has_basic"] & ~gtf_df["has_rubbish"]]
genes_basic_no_rubbish = set(basic_no_rubbish["gene_id"])
filters["Gene has >= 1 basic, non-rubbish transcript"] = gene_agg[
    gene_agg["gene_id"].isin(genes_basic_no_rubbish)
]
del filters["Gene has >= 1 basic AND no rubbish on basic"]

print(f"{'Filter':<50s} {'Genes':>8s} {'Transcripts':>13s}")
print("-" * 75)
for label, subset in filters.items():
    ng = len(subset)
    nt = subset["n_transcripts"].sum()
    print(f"{label:<50s} {ng:>8,} ({ng/n_genes:>5.1%}) {nt:>8,}")

Total lncRNA genes: 34,846

Filter                                                Genes   Transcripts
---------------------------------------------------------------------------
No filter (all)                                      34,846 (100.0%)  189,149
Gene has >= 1 basic                                  34,846 (100.0%)  189,149
Gene has >= 1 non-rubbish                            34,798 (99.9%)  189,094
Gene has >= 1 basic, non-rubbish transcript          34,797 (99.9%)  189,092


## 12. Transcript-level filter scenarios: how many transcripts survive?

In [16]:
lncrna_only = gtf_df[gtf_df["transcript_type"] == "lncRNA"]

scenarios = {
    "All transcripts from lncRNA genes":         gtf_df,
    "Only transcript_type == lncRNA":            lncrna_only,
    "Drop rubbish-tagged":                       gtf_df[~gtf_df["has_rubbish"]],
    "Keep basic only":                           gtf_df[gtf_df["has_basic"]],
    "Keep basic, drop rubbish":                  gtf_df[gtf_df["has_basic"] & ~gtf_df["has_rubbish"]],
    "Keep basic OR primary":                     gtf_df[gtf_df["has_basic"] | gtf_df["has_primary"]],
    "Keep basic OR primary, drop rubbish":       gtf_df[(gtf_df["has_basic"] | gtf_df["has_primary"]) & ~gtf_df["has_rubbish"]],
    "Keep level 1-2 only":                       gtf_df[gtf_df["level"].isin([1, 2])],
    "Keep level 1-2, basic, no rubbish":         gtf_df[gtf_df["level"].isin([1, 2]) & gtf_df["has_basic"] & ~gtf_df["has_rubbish"]],
    "Primary only (any biotype)":                gtf_df[gtf_df["has_primary"]],
    "Primary only (lncRNA biotype only)":        lncrna_only[lncrna_only["has_primary"]],
}

n_total_tx = len(gtf_df)
n_total_genes = gtf_df["gene_id"].nunique()

rows = []
for label, subset in scenarios.items():
    nt = len(subset)
    ng = subset["gene_id"].nunique()
    rows.append({
        "Filter": label,
        "Transcripts": f"{nt:,}",
        "% tx": f"{nt/n_total_tx:.1%}",
        "Genes": f"{ng:,}",
        "% genes": f"{ng/n_total_genes:.1%}",
    })

scenario_df = pd.DataFrame(rows).set_index("Filter")
scenario_df

,Transcripts,% tx,Genes,% genes
Filter,,,,
All lncRNA transcripts,"189,149",100.0%,"34,846",100.0%
Drop rubbish-tagged,"187,350",99.0%,"34,798",99.9%
Keep basic only,"69,521",36.8%,"34,846",100.0%
"Keep basic, drop rubbish","69,462",36.7%,"34,797",99.9%
Keep basic OR primary,"75,506",39.9%,"34,846",100.0%
"Keep basic OR primary, drop rubbish","75,358",39.8%,"34,797",99.9%
Keep level 1-2 only,"189,042",99.9%,"34,846",100.0%
"Keep level 1-2, basic, no rubbish","69,444",36.7%,"34,795",99.9%


## 13. All unique tags across lncRNA transcripts

In [17]:
all_tag_counts = Counter()
for tags in gtf_df["all_tags"]:
    all_tag_counts.update(tags)

tag_summary = pd.DataFrame(
    [(tag, cnt, f"{cnt/len(gtf_df):.1%}") for tag, cnt in all_tag_counts.most_common()],
    columns=["Tag", "Count", "% of lncRNA transcripts"],
)
tag_summary

,Tag,Count,% of lncRNA transcripts
0,TAGENE,162216,85.8%
1,basic,69521,36.8%
2,Ensembl_canonical,34846,18.4%
3,GENCODE_Primary,34846,18.4%
4,RNA_Seq_supported_only,5449,2.9%
5,exp_conf,2582,1.4%
6,nested_454_RNA_Seq_supported,1043,0.6%
7,mRNA_start_NF,909,0.5%
8,mRNA_end_NF,861,0.5%
9,454_RNA_Seq_supported,775,0.4%


## 14. Sanity check: H19, MEG3, GAS5

Verify these well-known lncRNA genes survive the "primary only" filter.
Some of their transcripts have biotype `retained_intron` rather than `lncRNA`.

In [ ]:
TARGET_GENES = ["H19", "MEG3", "GAS5"]

for gname in TARGET_GENES:
    sub = gtf_df[gtf_df["gene_name"] == gname]
    if sub.empty:
        print(f"\n{'='*60}")
        print(f"  {gname}: NOT FOUND in lncRNA genes")
        continue

    gid = sub["gene_id"].iloc[0]
    n_tx = len(sub)
    biotypes = sub["transcript_type"].value_counts()
    primary = sub[sub["has_primary"]]

    print(f"\n{'='*60}")
    print(f"  {gname} ({gid})")
    print(f"  Total transcripts: {n_tx}")
    print(f"  Biotype breakdown:")
    for bt, cnt in biotypes.items():
        print(f"    {bt}: {cnt}")

    if primary.empty:
        print(f"  PRIMARY transcript: NONE — gene would be LOST")
    else:
        for _, row in primary.iterrows():
            print(f"  PRIMARY transcript: {row['transcript_id']}")
            print(f"    biotype:  {row['transcript_type']}")
            print(f"    basic:    {row['has_basic']}")
            print(f"    level:    {row['level']}")

    in_primary_filter = not primary.empty
    in_lncrna_primary = not primary.empty and all(
        primary["transcript_type"] == "lncRNA"
    )
    print(f"  Survives 'primary only (any biotype)':       {in_primary_filter}")
    print(f"  Survives 'primary only (lncRNA biotype only)': {in_lncrna_primary}")

---

## 15. Export: primary-only lncRNA annotation

Keep all non-lncRNA transcripts unchanged; for lncRNA **genes** (by `gene_type`)
keep only the GENCODE Primary transcript (one per gene), regardless of transcript biotype.
Writes a BED12 and matching metadata TSV.

In [ ]:
from pyrion.core.genes import TranscriptsCollection
from pyrion.ops.transcript_serialization import save_transcripts_collection_to_bed12

OUT_BED = "reference_annotation/hg38.primary_only.bed"
OUT_META = "reference_annotation/hg38.primary_only.transcript_metadata.tsv"

# All lncRNA gene IDs (by gene_type, not transcript_type)
lncrna_gene_ids = set(gtf_df["gene_id"])
# Primary transcript IDs for lncRNA genes (one per gene, any biotype)
lncrna_primary_tids = set(
    gtf_df.loc[gtf_df["has_primary"], "transcript_id"]
)

kept = []
kept_meta = []

for t in transcripts:
    biotype = gene_data.get_transcript_biotype(t.id)
    gid = gene_data.get_gene(t.id)
    if gid in lncrna_gene_ids:
        if t.id in lncrna_primary_tids:
            kept.append(t)
            kept_meta.append((t.id, gid or t.id, biotype or "lncRNA"))
    else:
        kept.append(t)
        kept_meta.append((t.id, gid or t.id, biotype or "unknown"))

save_transcripts_collection_to_bed12(TranscriptsCollection(kept), OUT_BED)

with open(OUT_META, "w") as f:
    f.write("transcript_id\tgene_id\ttranscript_biotype\n")
    for tid, gid, bt in kept_meta:
        f.write(f"{tid}\t{gid}\t{bt}\n")

n_from_lnc_genes = sum(1 for t, g, _ in kept_meta if g in lncrna_gene_ids)
n_other = len(kept_meta) - n_from_lnc_genes
print(f"Written {len(kept):,} transcripts to {OUT_BED}")
print(f"  non-lncRNA genes (unchanged): {n_other:,}")
print(f"  lncRNA genes (primary only):  {n_from_lnc_genes:,}")
print(f"Metadata: {OUT_META}")